In [1]:
import gridlabd

In [2]:
dir(gridlabd)

['ConfigurationError',
 'GLDCheckPointMode',
 'GLDErrorCode',
 'GridLABDError',
 'GridLabD',
 'Path',
 'Simulation',
 'SimulationError',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 '_lib_dir',
 '_package_dir',
 '_share_dir',
 'bundle_utils',
 'get_gridlabd_info',
 'glpath',
 'glpath_components',
 'gridlabd_core',
 'hello',
 'info',
 'load_model',
 'os',
 'path_sep',
 'root',
 'setup_bundled_environment',
 'simulation',
 'version']

In [3]:
gld = gridlabd.GridLabD()

In [4]:
from pathlib import Path
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld.set_working_directory(str(model_dir))


Working directory set to: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests


GLDErrorCode.SUCCESS

In [5]:
gld.set_config_file("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/gridlabd.conf")

Setting config file: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/gridlabd.conf


GLDErrorCode.SUCCESS

In [1]:
# Let's check the GridLAB-D installation paths
import gridlabd
import os

print("GridLAB-D package location:", gridlabd.__file__)
print("Package directory:", gridlabd._package_dir)
print("Lib directory:", gridlabd._lib_dir)
print("Share directory:", gridlabd._share_dir)

# Check if the library files exist
lib_path = os.path.join(gridlabd._package_dir, 'libgldapi.so')
print(f"\nlibgldapi.so exists: {os.path.exists(lib_path)}")
if os.path.exists(lib_path):
    print(f"  Size: {os.path.getsize(lib_path)} bytes")

# Check the lib directory
if os.path.exists(gridlabd._lib_dir):
    print(f"\nModules in lib directory:")
    for f in os.listdir(gridlabd._lib_dir):
        if f.endswith('.so'):
            print(f"  - {f}")
else:
    print(f"\nLib directory doesn't exist: {gridlabd._lib_dir}")

GridLAB-D package location: /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/__init__.py
Package directory: /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd
Lib directory: /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/lib
Share directory: /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/share

libgldapi.so exists: True
  Size: 6234064 bytes

Modules in lib directory:
  - assert.so
  - climate.so
  - commercial.so
  - connection.so
  - generators.so
  - glsolvers.so
  - glxengine.so
  - libgldapi.so
  - market.so
  - mysql.so
  - optimize.so
  - powerflow.so
  - reliability.so
  - residential.so
  - tape.so
  - tape_file.so
  - tape_plot.so


In [2]:
# Try a simpler test first - just check if the C++ module loads
try:
    import gridlabd.gridlabd_core as core
    print("C++ core module loaded successfully")
    print(f"Core module location: {core.__file__}")
    
    # Try the hello function (doesn't require GridLAB-D initialization)
    result = core.hello()
    print(f"Hello result: {result}")
except Exception as e:
    print(f"Error loading core: {e}")
    import traceback
    traceback.print_exc()

C++ core module loaded successfully
Core module location: /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/gridlabd_core.cpython-312-x86_64-linux-gnu.so
Hello result: Hello from GridLAB-D Python bindings!


## Diagnosis: Kernel Crash Investigation

The kernel crashes when calling `load_glm()`. This is likely due to one of these issues:

1. **Missing shared library dependencies** - libgldapi.so might depend on other system libraries
2. **Incorrect RPATH** - The module can't find libgldapi.so at runtime
3. **Binary incompatibility** - The wheel was built on a different system with different glibc/libraries

Let's investigate each possibility:

In [3]:
# Check shared library dependencies
import subprocess
import gridlabd
import os

# Find the gridlabd_core module
core_path = None
package_dir = gridlabd._package_dir
for f in os.listdir(package_dir):
    if f.startswith('gridlabd_core') and f.endswith('.so'):
        core_path = os.path.join(package_dir, f)
        break

if core_path:
    print(f"Found core module: {core_path}\n")
    
    # Check dependencies with ldd
    result = subprocess.run(['ldd', core_path], capture_output=True, text=True)
    print("Shared library dependencies:")
    print(result.stdout)
    
    if 'not found' in result.stdout:
        print("\n⚠️  WARNING: Some libraries are missing!")
else:
    print("Could not find gridlabd_core module")

Found core module: /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/gridlabd_core.cpython-312-x86_64-linux-gnu.so

Shared library dependencies:
	linux-vdso.so.1 (0x00007ffed226d000)
	libgldapi.so => /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/libgldapi.so (0x000076d6c6a30000)
	libstdc++.so.6 => /lib/x86_64-linux-gnu/libstdc++.so.6 (0x000076d6c6600000)
	libgcc_s.so.1 => /lib/x86_64-linux-gnu/libgcc_s.so.1 (0x000076d6c69fc000)
	libc.so.6 => /lib/x86_64-linux-gnu/libc.so.6 (0x000076d6c6200000)
	libncurses.so.6 => /lib/x86_64-linux-gnu/libncurses.so.6 (0x000076d6c69d3000)
	libtinfo.so.6 => /lib/x86_64-linux-gnu/libtinfo.so.6 (0x000076d6c699d000)
	libm.so.6 => /lib/x86_64-linux-gnu/libm.so.6 (0x000076d6c68b4000)
	/lib64/ld-linux-x86-64.so.2 (0x000076d6c7427000)



### Diagnosis Results

✅ Package installed correctly  
✅ C++ module loads  
✅ All dependencies found  
✅ RPATH working

**The crash happens inside GridLAB-D's load_glm() function.** This might be due to:
- Signal handlers conflicting with Jupyter
- Improper error handling in the C++ code
- GridLAB-D expecting a terminal environment

Let's try to catch the crash with a simpler test:

In [4]:
# Test if this is a signal handling issue
# Try running in a subprocess to isolate the crash
import subprocess
import sys

test_script = """
import gridlabd
from pathlib import Path

gld = gridlabd.GridLabD()
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld.set_working_directory(str(model_dir))
print("Working directory set")

result = gld.load_glm(["gridlabd", "./test_HVAC_balance.glm"])
print(f"Load result: {result}")
"""

result = subprocess.run(
    [sys.executable, "-c", test_script],
    capture_output=True,
    text=True,
    timeout=10
)

print("STDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)
print(f"\nReturn code: {result.returncode}")

STDOUT:
Working directory set to: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests
Working directory set


STDERR:

ERROR    [INIT] : /mnt/c/Projects/Gridlab-d/gridlab-d/gldcore/module.cpp(547): module 'residential' load failed - residential.so: cannot open shared object file: No such file or directory
./test_HVAC_balance.glm(7): module 'residential' load failed, No such file or directory
./test_HVAC_balance.glm(7): load failed at or near 'module resid...'
ERROR    [INIT] : unable to load './test_HVAC_balance.glm': No such file or directory
FATAL    [INIT] : shutdown after command line rejected


Return code: 1


In [5]:
# Check what environment variables are set
import os
import gridlabd

print("Environment variables relevant to GridLAB-D:")
for key in ['GLPATH', 'GLD_MODULE_PATH', 'LD_LIBRARY_PATH']:
    value = os.environ.get(key, 'NOT SET')
    print(f"  {key}: {value}")

print(f"\nPackage _lib_dir: {gridlabd._lib_dir}")
print(f"Package _share_dir: {gridlabd._share_dir}")

# Check if residential.so exists in _lib_dir
import os
residential_path = os.path.join(gridlabd._lib_dir, 'residential.so')
print(f"\nresidential.so exists at {residential_path}: {os.path.exists(residential_path)}")

Environment variables relevant to GridLAB-D:
  GLPATH: /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/share:/mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/lib
  GLD_MODULE_PATH: NOT SET
  LD_LIBRARY_PATH: NOT SET

Package _lib_dir: /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/lib
Package _share_dir: /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/share

residential.so exists at /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/lib/residential.so: True


### Solution: Set GLPATH Manually

The issue is that GridLAB-D can't find the module libraries. We need to explicitly add the lib directory to GLPATH:

In [1]:
# WORKAROUND: Manually set GLPATH to include the lib directory
import os
import gridlabd
from pathlib import Path

# Make sure GLPATH includes both share and lib directories
glpath_parts = []
if 'GLPATH' in os.environ:
    glpath_parts.append(os.environ['GLPATH'])

glpath_parts.append(str(gridlabd._share_dir))
glpath_parts.append(str(gridlabd._lib_dir))

os.environ['GLPATH'] = ':'.join(glpath_parts)
print(f"GLPATH set to: {os.environ['GLPATH']}")

# Now try loading again
gld_fixed = gridlabd.GridLabD()
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld_fixed.set_working_directory(str(model_dir))
'''
result = gld_fixed.load_glm(["gridlabd", "./test_HVAC_balance.glm"])
print(f"\nLoad result: {result}")

if result == gridlabd.GLDErrorCode.SUCCESS:
    print("✅ Successfully loaded the model!")
    result = gld_fixed.run()
    print(f"Run result: {result}")
else:
    print(f"❌ Load failed with error: {result}") '''

GLPATH set to: /mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/share:/mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/lib:/mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/share:/mnt/c/Projects/Gridlab-d/gridlab-d/testenv/lib/python3.12/site-packages/gridlabd/lib
Working directory set to: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests


'\nresult = gld_fixed.load_glm(["gridlabd", "./test_HVAC_balance.glm"])\nprint(f"\nLoad result: {result}")\n\nif result == gridlabd.GLDErrorCode.SUCCESS:\n    print("✅ Successfully loaded the model!")\n    result = gld_fixed.run()\n    print(f"Run result: {result}")\nelse:\n    print(f"❌ Load failed with error: {result}") '

In [ ]:
result = gld_fixed.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

: 

## Root Cause: Binary Compatibility Issue

The kernel crashes because:
1. ✅ GLPATH is set correctly  
2. ✅ All libraries exist
3. ❌ **The binary was built in `.venv` but you're running it in `testenv`**

The wheel you uploaded to test.pypi.org contains compiled C++ code that may have hardcoded paths or dependencies from the build environment.

### Solution: Test with the locally-built development version

Let's compare with the version installed in editable mode from your dev environment:

In [ ]:
gld.load_glm(["gridlabd", "./test_HVAC_balance.glm", "--verbose"])

: 

In [ ]:
gld.run()

In [ ]:
# Don't call exit_gld() in notebooks - it crashes the kernel
# Just let Python clean up automatically
del gld

## Testing More Functionality

In [3]:
# Create a new instance for testing
import json
from pathlib import Path
gld2 = gridlabd.GridLabD()
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld2.set_working_directory(str(model_dir))


Working directory set to: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests


GLDErrorCode.SUCCESS

In [ ]:
del gld2

In [4]:

gld2.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

GLDErrorCode.SUCCESS

In [5]:
gld2.run()


Using previous start_time: 0.00
Using previous stop_time: 0.00

WARNING  [INIT] : Daylight saving time (DST) is not handled correctly when using TMY2 datasets; please use TMY3 for DST-corrected weather data.


GLDErrorCode.SUCCESS

In [6]:

# Get checkpoint as JSON string
checkpoint_json = gld2.get_checkpoint_json()
checkpoint_data = json.loads(checkpoint_json)

print("Checkpoint keys:", list(checkpoint_data.keys())[:10])  # First 10 keys
print(f"Total objects: {len(checkpoint_data)}")

Checkpoint keys: ['__preamble', 'clock', 'objects']
Total objects: 3


In [7]:
list(checkpoint_data.keys())

['__preamble', 'clock', 'objects']

In [13]:
checkpoint_data['__preamble']

{'comments': ['// GridLAB-D checkpoint data export',
  '// Generated at timestamp: 989150402',
  '// Checkpoint sequence: 0']}

In [14]:
checkpoint_data['clock']

{'starttime': 989064000,
 'stoptime': 989150402,
 'timestamp': 989150402,
 'timezone': 'PST8PDT'}

In [17]:
list(checkpoint_data['objects'])

['climate', 'house', 'recorder', 'triplex_meter']

In [20]:
# Get current simulation time
status, current_time = gld2.get_time()
print(f"Status: {status}")
print(f"Current simulation time: {current_time}")

Status: GLDErrorCode.SUCCESS
Current simulation time: 2025-06-12T12:00:00
Getting current time: 2025-06-12T12:00:00


In [ ]:
gld2.step()

: 

In [ ]:

# Get checkpoint as JSON string...again
# Fix in python???
checkpoint_json2 = gld2.get_checkpoint_json()
checkpoint_data2 = json.loads(checkpoint_json2)

print("Checkpoint keys:", list(checkpoint_data2.keys())[:10])  # First 10 keys
print(f"Total objects: {len(checkpoint_data2)}")

AttributeError: 'NoneType' object has no attribute 'keys'

In [22]:
# Debug: Check what checkpoint_json2 actually contains
print("checkpoint_json2 type:", type(checkpoint_json2))
print("checkpoint_json2 length:", len(checkpoint_json2))
print("checkpoint_json2 content (first 200 chars):", checkpoint_json2[:200])
print("checkpoint_json2 content (last 200 chars):", checkpoint_json2[-200:])

checkpoint_json2 type: <class 'str'>
checkpoint_json2 length: 4
checkpoint_json2 content (first 200 chars): null
checkpoint_json2 content (last 200 chars): null


In [ ]:
# Check checkpoint configuration (these are global variables in GridLAB-D)
# You can access them via the GLM command line arguments or config

# For now, the simplest solution is to:
# 1. Call get_checkpoint_json() once after run() and save the result
# 2. Reuse that data instead of calling again

# OR advance the simulation time significantly between calls:
# - For CPT_SIM (default): advance simulation by 86400+ seconds (1 day)
# - For CPT_WALL: wait 3600+ real seconds (not practical!)

print("Best practice: Cache the first checkpoint result and reuse it")
print("The checkpoint data doesn't change unless you run/step the simulation further")

### Understanding Checkpoint Intervals

GridLAB-D uses global variables to control checkpoint behavior:
- `checkpoint_type`: NONE (0), WALL (1), or SIM (2)
- `checkpoint_interval`: Time between checkpoints (seconds for WALL, simulation seconds for SIM)
  - Default for WALL: 3600 seconds (1 hour)
  - Default for SIM: 86400 seconds (1 day simulation time)

In [ ]:
# Run simulation step by step
gld3 = gridlabd.GridLabD()
gld3.set_working_directory(str(model_dir))
gld3.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

print("Starting step-by-step simulation...")
for i in range(5):  # Run 5 steps
    sim_time = gld3.step()
    status, time_str = gld3.get_time()
    print(f"Step {i+1}: time = {sim_time} ({time_str})")
    
print("Step simulation complete")

In [ ]:
# Use the higher-level Simulation wrapper
from gridlabd import Simulation

with Simulation() as sim:
    sim.gld.set_working_directory(str(model_dir))
    result = sim.gld.load_glm(["gridlabd", "./test_HVAC_balance.glm"])
    print(f"Load result: {result}")
    
    result = sim.gld.run()
    print(f"Run result: {result}")
    
    # Get checkpoint data through the wrapper
    checkpoint = sim.gld.get_checkpoint_json()
    data = json.loads(checkpoint)
    print(f"Objects in simulation: {len(data)}")

In [ ]:
# Get installation paths
print("GridLAB-D Info:")
print(f"  Version: {gridlabd.version()}")
print(f"  Install root: {gridlabd.GridLabD.get_install_root()}")
print(f"  Executable: {gridlabd.GridLabD.get_executable_path()}")
print(f"  Hello: {gridlabd.hello()}")

In [ ]:
# Run with specific start/stop times
gld4 = gridlabd.GridLabD()
gld4.set_working_directory(str(model_dir))
gld4.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

# Run for a specific time range (example timestamps)
# Note: times are in seconds since epoch
result = gld4.run(start_time=0.0, stop_time=3600.0)  # Run for 1 hour
print(f"Simulation result: {result}")

status, final_time = gld4.get_time()
print(f"Final time: {final_time}")

In [ ]:
# NOTE: get_checkpoint_json() uses an internal checkpoint interval
# Calling it twice rapidly may return empty data the second time
# Solution: Either reuse the first data or pass "" to force a fresh checkpoint

# Get checkpoint with explicit empty path to force fresh data
checkpoint_json2 = gld2.get_checkpoint_json("")
checkpoint_data2 = json.loads(checkpoint_json2) if checkpoint_json2 else None

if checkpoint_data2:
    print("Checkpoint keys:", list(checkpoint_data2.keys())[:10])  # First 10 keys
    print(f"Total objects: {len(checkpoint_data2)}")
else:
    print("No checkpoint data returned (interval not met). Reusing original checkpoint_data.")
    checkpoint_data2 = checkpoint_data
if load_result == gridlabd.GLDErrorCode.SUCCESS:
    run_result = gld5.run()
    print(f"Run result: {run_result}")
    
    # Get some data
    checkpoint = gld5.get_checkpoint_json()
    data = json.loads(checkpoint)
    print(f"Successfully simulated {len(data)} objects")

In [ ]:
# Analyze checkpoint data with pandas (if available)
try:
    import pandas as pd
    
    # Convert checkpoint to DataFrame for analysis
    checkpoint_json = gld5.get_checkpoint_json()
    data = json.loads(checkpoint_json)
    
    # Create a summary DataFrame
    summary = []
    for obj_name, obj_data in list(data.items())[:20]:  # First 20 objects
        summary.append({
            'name': obj_name,
            'class': obj_data.get('class', 'unknown'),
            'parent': obj_data.get('parent', 'none'),
            'rank': obj_data.get('rank', -1)
        })
    
    df = pd.DataFrame(summary)
    print("Object Summary:")
    print(df)
    print(f"\nClass distribution:\n{df['class'].value_counts()}")
    
except ImportError:
    print("pandas not available, skipping DataFrame analysis")

### 8. Analyzing Checkpoint Data with Pandas

### 7. Using the Convenience Function

### 6. Running with Custom Start/Stop Times

### 5. Query GridLAB-D Installation Info

### 4. Using the High-Level Simulation Class

### 3. Step-by-Step Simulation

### 2. Query Simulation Time

In [ ]:
# Inspect a specific object from the checkpoint
if len(checkpoint_data) > 0:
    first_obj_name = list(checkpoint_data.keys())[0]
    print(f"\nFirst object: {first_obj_name}")
    print(json.dumps(checkpoint_data[first_obj_name], indent=2)[:500])  # First 500 chars

### 1. Get Checkpoint JSON (Model State)

## Testing the Fixed Wheel with RPATH

After discovering the issue was broken RPATH in the module `.so` files, we've rebuilt the wheel with proper RPATH fixes using `patchelf`. Let's verify the fix works.

In [1]:
# First, verify RPATH in the newly built wheel
import subprocess
import os
import zipfile

wheel_path = "/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/dist/gridlabd-5.0.0-cp312-cp312-manylinux_2_39_x86_64.whl"

# Extract and check
extract_dir = "/tmp/test_wheel_rpath"
os.makedirs(extract_dir, exist_ok=True)

# Unzip the wheel using Python's zipfile module
print(f"Extracting wheel: {wheel_path}")
with zipfile.ZipFile(wheel_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print("✓ Extracted")

# Check RPATH of residential.so
residential_path = f"{extract_dir}/gridlabd/lib/residential.so"
result = subprocess.run(["readelf", "-d", residential_path], capture_output=True, text=True)

print("\nRPATH/RUNPATH in FIXED residential.so:")
for line in result.stdout.split('\n'):
    if 'RPATH' in line or 'RUNPATH' in line:
        print(f"  {line.strip()}")
        
# Also check libgldapi.so
libgldapi_path = f"{extract_dir}/gridlabd/libgldapi.so"
result2 = subprocess.run(["readelf", "-d", libgldapi_path], capture_output=True, text=True)

print("\nRPATH/RUNPATH in FIXED libgldapi.so:")
for line in result2.stdout.split('\n'):
    if 'RPATH' in line or 'RUNPATH' in line:
        print(f"  {line.strip()}")

Extracting wheel: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/dist/gridlabd-5.0.0-cp312-cp312-manylinux_2_39_x86_64.whl
✓ Extracted

RPATH/RUNPATH in FIXED residential.so:
  0x000000000000001d (RUNPATH)            Library runpath: [$ORIGIN:$ORIGIN/..]

RPATH/RUNPATH in FIXED libgldapi.so:
  0x000000000000001d (RUNPATH)            Library runpath: [$ORIGIN]


In [1]:
# Install the fixed wheel (uninstall old version first)
!python3 -m pip uninstall -y gridlabd
!python3 -m pip install /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/dist/gridlabd-5.0.0-cp312-cp312-manylinux_2_39_x86_64.whl

Found existing installation: gridlabd 5.0.0
Uninstalling gridlabd-5.0.0:
  Successfully uninstalled gridlabd-5.0.0
Uninstalling gridlabd-5.0.0:
  Successfully uninstalled gridlabd-5.0.0
Processing /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/dist/gridlabd-5.0.0-cp312-cp312-manylinux_2_39_x86_64.whl
Processing /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/dist/gridlabd-5.0.0-cp312-cp312-manylinux_2_39_x86_64.whl


In [2]:
# Test basic import with the fixed wheel
import gridlabd
print(f"✓ gridlabd imported successfully")
print(f"  Version: {gridlabd.version()}")
print(f"  Hello: {gridlabd.hello()}")

✓ gridlabd imported successfully
  Version: 5.0.0
  Hello: Hello from GridLAB-D Python bindings!


In [ ]:
# THE CRITICAL TEST: Load a GLM file (this requires residential.so to load properly)
# Run in subprocess to avoid kernel crash and see actual error
import subprocess
import sys

test_script = """
import gridlabd

gld = gridlabd.GridLabD()
gld.set_working_directory("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests")

print("Attempting to load GLM file with residential module...")
result = gld.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

if result == gridlabd.GLDErrorCode.SUCCESS:
    print(f"✓✓✓ SUCCESS! GLM loaded successfully!")
    print(f"  residential.so loaded correctly with FIXED EXECUTE PERMISSIONS!")
    
    # Try running the simulation too
    print("\\nRunning simulation...")
    run_result = gld.run()
    if run_result == gridlabd.GLDErrorCode.SUCCESS:
        print(f"✓✓✓ SIMULATION COMPLETED SUCCESSFULLY!")
        print(f"\\nThe wheel is FIXED and ready for PyPI upload!")
    else:
        print(f"✗ Simulation failed with error code: {run_result}")
else:
    print(f"✗ FAILED: GLM loading failed with error code: {result}")
"""

result = subprocess.run(
    [sys.executable, "-c", test_script],
    capture_output=True,
    text=True,
    timeout=30
)

print("STDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)
print(f"\nReturn code: {result.returncode}")


STDOUT:
Working directory set to: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests
Attempting to load GLM file with residential module...


STDERR:

ERROR    [INIT] : /mnt/c/Projects/Gridlab-d/gridlab-d/gldcore/module.cpp(547): module 'residential' load failed - residential.so: cannot open shared object file: No such file or directory
./test_HVAC_balance.glm(7): module 'residential' load failed, No such file or directory
./test_HVAC_balance.glm(7): load failed at or near 'module resid...'
ERROR    [INIT] : unable to load './test_HVAC_balance.glm': No such file or directory
FATAL    [INIT] : shutdown after command line rejected


Return code: 1


In [ ]:
# Run the simulation to completion
if result == gridlabd.GLDErrorCode.SUCCESS:
    print("Running simulation...")
    run_result = gld.run()
    if run_result == gridlabd.GLDErrorCode.SUCCESS:
        print(f"✓✓✓ SIMULATION COMPLETED SUCCESSFULLY!")
        print(f"\nThe wheel is FIXED and ready for PyPI upload!")
    else:
        print(f"✗ Simulation failed with error code: {run_result}")